# Pitch Perfect — Gemini Multimodal Prototype

Run this in Google Colab (has internet access, unlike a sandboxed dev environment).
This notebook validates the **core AI piece** of Pitch Perfect before you wire up the full FastAPI/React app:

1. Upload a short pitch video (or slides + audio)
2. Send it to Gemini with a structured `response_schema`
3. Get back a 5-axis scorecard as clean JSON
4. Ask a grounded follow-up question about the pitch

**You need:** a Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey) (free tier is fine to start).

> Note: Google's SDK method names/parameters shift over time. If a call below errors, check the current `google-genai` docs — the *shape* of this pipeline (upload → multimodal call → schema'd JSON) will still be right.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import getpass
import os

# Paste your key when prompted (never hardcode it in the notebook)
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"  # multimodal, fast, cheap — good fit for this use case

## 1. Upload your pitch video

Keep it under ~3 minutes for the free tier — long video processing is slow and eats quota fast.

In [ ]:
from google.colab import files

print("Upload a short pitch video (mp4/mov, ideally under 3 min)")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

In [ ]:
# Upload the file to Gemini's file store (required for video/audio inputs)
video_file = client.files.upload(file=video_path)

# Wait for processing to finish
import time
while video_file.state.name == "PROCESSING":
    print("Processing video...")
    time.sleep(5)
    video_file = client.files.get(name=video_file.name)

if video_file.state.name == "FAILED":
    raise ValueError("Video processing failed")

print(f"Ready: {video_file.name}")

## 2. Define the 5-axis scorecard schema

This is the key design decision from your resume bullet: forcing Gemini into a **structured schema** (not free text) makes scoring deterministic and easy to render in the UI, and — combined with the citation field below — gives you the "grounded, timestamped evidence" behavior that reduces hallucinated feedback.

In [ ]:
scorecard_schema = {
    "type": "OBJECT",
    "properties": {
        "axes": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "name": {
                        "type": "STRING",
                        "enum": ["Clarity", "Pacing", "Technical Depth", "Storytelling", "Slide Quality"]
                    },
                    "score": {"type": "INTEGER", "description": "Score from 1-10"},
                    "critiques": {
                        "type": "ARRAY",
                        "items": {
                            "type": "OBJECT",
                            "properties": {
                                "note": {"type": "STRING"},
                                "timestamp_seconds": {"type": "INTEGER", "description": "Where in the video this occurs, for click-to-jump"}
                            },
                            "required": ["note", "timestamp_seconds"]
                        }
                    },
                    "rewrite_suggestion": {"type": "STRING"}
                },
                "required": ["name", "score", "critiques", "rewrite_suggestion"]
            }
        },
        "overall_summary": {"type": "STRING"}
    },
    "required": ["axes", "overall_summary"]
}

## 3. Run the multimodal analysis call

In [ ]:
SCORECARD_PROMPT = """You are an expert hackathon/pitch coach reviewing a student's pitch video.

Watch and listen to the full video. Score it on exactly these 5 axes: Clarity, Pacing, Technical Depth, Storytelling, Slide Quality.

For each axis:
- Give a score from 1-10
- Give exactly 2 specific critiques, each grounded to a real timestamp in the video (timestamp_seconds must be an actual moment you observed, not a guess)
- Give 1 concrete rewrite suggestion

Be specific and honest — reference what was actually said or shown, not generic pitch advice. Then write a 2-3 sentence overall_summary.
"""

response = client.models.generate_content(
    model=MODEL,
    contents=[video_file, SCORECARD_PROMPT],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=scorecard_schema,
    ),
)

import json
scorecard = json.loads(response.text)
print(json.dumps(scorecard, indent=2))

## 4. Pretty-print the scorecard (sanity check before you build the React UI)

In [ ]:
print(f"OVERALL: {scorecard['overall_summary']}\n")
for axis in scorecard["axes"]:
    print(f"{axis['name']}: {axis['score']}/10")
    for c in axis["critiques"]:
        print(f"  - [{c['timestamp_seconds']}s] {c['note']}")
    print(f"  Rewrite: {axis['rewrite_suggestion']}\n")

## 5. "Ask my pitch" — grounded chat

This is the second AI feature from your resume bullet: a chat grounded on the user's own pitch content, e.g. "What would a Google engineer ask me here?"

In [ ]:
def ask_my_pitch(question: str):
    grounding_prompt = f"""You have already watched the attached pitch video and produced this scorecard:
{json.dumps(scorecard)}

Answer the user's question ONLY using what you observed in the video and the scorecard above. If asked to predict questions an interviewer might ask, ground them in specific, real moments from the pitch (reference timestamps).

User question: {question}"""

    resp = client.models.generate_content(
        model=MODEL,
        contents=[video_file, grounding_prompt],
    )
    return resp.text

print(ask_my_pitch("What would a Google engineer ask me after this pitch?"))

## Next steps

Once this notebook produces a real scorecard you're happy with:
1. Copy `scorecard_schema` and `SCORECARD_PROMPT` into `backend/gemini_service.py` in the full project (unchanged — they're written to drop straight in).
2. Tune the prompt based on what you see here — this is the cheapest place to iterate, before it's wired into the app.
3. Move to the FastAPI backend to wrap this in a real upload → analyze → store flow.